# SI Figure S13: MagNET-x vs DFT explicit-solvent corrections

Four panels, both nuclei: **A** MagNET-x (NN) vs DFT explicit-solvent correction per site/solvent,
Desmond and OpenMM overlaid; **B** the NN-minus-DFT error distribution; **C** every site's DFT and NN
correction in chloroform, one column per solute; **D** the semi-parsimonious composite model's fit accuracy vs experiment across the four OpenMM solvents, split by engine and
DFT-vs-NN source.

In [ ]:
import os, sys

# make the in-repo modules importable (not pip-installed)
REPO = os.path.abspath("../..")
for _p in ("data/delta22", "analysis/code", "analysis/code/shared"):
    sys.path.insert(0, os.path.join(REPO, _p))

In [ ]:
import matplotlib.pyplot as plt

import delta22
import delta22_plots
import paths

In [ ]:
DELTA22_HDF5 = paths.dataset_file("delta22", root=REPO)
XLSX = os.path.join(REPO, "data", "delta22", "delta22_experimental.xlsx")

def figure_path(name):
    os.makedirs("figures", exist_ok=True)
    return os.path.join("figures", name)

In [ ]:
dft = delta22.load_query_df_dft(DELTA22_HDF5, XLSX, verbose=False)
nn = delta22.load_query_df_nn(DELTA22_HDF5, XLSX, verbose=False)

In [ ]:
# openMM/Desmond engine hues (used by the scatter/histogram/site/fitting-accuracy panels below)
ENGINE_COLORS = {"openMM": "#2E86AB", "desmond": "#A23B72"}
ENGINE_LABELS = {"desmond": "Desmond", "openMM": "OpenMM"}

## Panels A and B: MagNET-x vs DFT explicit corrections, Desmond and OpenMM overlaid

In [ ]:
explicit_by_engine = delta22.compare_dft_nn_by_engine(dft, nn, keys=("solute", "site", "nucleus", "solvent"))
delta22_plots.plot_dft_nn_scatter_by_engine(explicit_by_engine, ENGINE_COLORS, ENGINE_LABELS,
                                            save_path=figure_path("si_figure_s13a_scatter.png"))
delta22_plots.plot_dft_nn_error_histogram_by_engine(explicit_by_engine, ENGINE_COLORS, ENGINE_LABELS,
                                                    save_path=figure_path("si_figure_s13b_hist.png"))
plt.show()

## Panel C: every proton/carbon site's explicit correction in chloroform

In [ ]:
for nucleus, label in [("H", "Proton"), ("C", "Carbon")]:
    pairs = delta22.explicit_correction_dft_nn_pairs(dft, nn, "chloroform", nucleus)
    delta22_plots.plot_explicit_correction_by_site(
        pairs, f"All {label} Sites: DFT vs NN Explicit Corrections (chloroform)", ENGINE_COLORS,
        save_path=figure_path(f"si_figure_s13c_sites_{'1H' if nucleus == 'H' else '13C'}.png"))
plt.show()

## Panel D: fitting accuracy of the semi-parsimonious composite model

In [ ]:
# the published panels use 250 seeded train/test splits
N_SPLITS = 250

In [ ]:
SOLVENT_ORDER = ["chloroform", "methanol", "TIP4P", "benzene"]
solutes = sorted(dft["solute"].unique())
for nucleus, label in [("H", "1H"), ("C", "13C")]:
    fitting = delta22.si_s13d_fitting_accuracy(dft, nn, SOLVENT_ORDER, N_SPLITS, solutes, nucleus=nucleus)
    delta22_plots.plot_fitting_accuracy_boxplot(
        fitting, SOLVENT_ORDER, label, ENGINE_COLORS,
        save_path=figure_path(f"si_figure_s13d_fitting_{'1H' if nucleus == 'H' else '13C'}.png"))
plt.show()